# 04 — Federated training: the five-rung ladder

Trains and compares, on identical data and the same small model:

| rung | who trains | why it's here |
|---|---|---|
| naive / seasonal-naive | nobody | the bar any model must beat |
| **local-only** | each trust alone | what a trust gets *without* the federation |
| **FedAvg** | all 35, weights only | the headline — no raw data leaves a trust |
| **FedAvg + fine-tune** | federation, then each trust personalizes | usually the practical winner |
| centralized | one pooled model | hypothetical upper bound (needs data sharing) |

**Choices:** model = specialty embedding (4) + 8 features → 32 → 32 → 1 MLP (~1.8k params —
small because 33k rows; bigger would overfit). The MLP predicts the **residual on top of last
month** (`output = lag1 + MLP(...)`): monthly demand is highly persistent, so the model *starts
at* the naive baseline and learns corrections. Trained with **L1 loss** so the training objective
matches the MAE metric (v1 used MSE + level-prediction and every learned model lost to naive —
see WORKLOG). No trust ID input — per-series normalization already absorbs scale; personalization
comes from fine-tuning. FedAvg is written manually (~15 lines) so every step is inspectable; each
round = broadcast → 2 local epochs per client → sample-weighted average. Model selection on
**val** (mean per-trust MAE), **test** (final 12 months) touched once per rung. MAE in real
referral units. Single seed here; multi-seed sweep is A100 work later.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

f = pd.read_csv("../data/processed/features.csv")
FEATS = ["lag1", "lag2", "lag3", "lag12", "backlog", "throughput", "month_sin", "month_cos"]
specs = sorted(f["specialty_code"].unique())
f["spec"] = f["specialty_code"].map({s: i for i, s in enumerate(specs)})
trusts = sorted(f["provider_code"].unique())
splits = {s: f[f["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}
device = "cuda" if torch.cuda.is_available() else "cpu"

def tensors(rows):
    return (torch.tensor(rows[FEATS].values, dtype=torch.float32).to(device),
            torch.tensor(rows["spec"].values).to(device),
            torch.tensor(rows["y"].values, dtype=torch.float32).to(device))

class Net(nn.Module):
    def __init__(self, emb=4, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(len(specs), emb)
        self.mlp = nn.Sequential(nn.Linear(len(FEATS) + emb, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, s):  # residual on lag1 (x[:, 0]) -> the model STARTS at the naive baseline
        return x[:, 0] + self.mlp(torch.cat([x, self.emb(s)], dim=1)).squeeze(-1)

def train_epochs(model, rows, epochs, lr=1e-3, batch=256):
    x, s, y = tensors(rows)
    dl = DataLoader(TensorDataset(x, s, y), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for xb, sb, yb in dl:
            opt.zero_grad()
            nn.functional.l1_loss(model(xb, sb), yb).backward()  # L1 = same shape as the MAE metric
            opt.step()
    return model

@torch.no_grad()
def predict_demand(model, rows):  # invert per-series normalization -> real referral units
    x, s, _ = tensors(rows)
    model.eval()
    ylog = model(x, s).cpu().numpy() * rows["sd"].values + rows["mu"].values
    return np.expm1(ylog)

def mae(rows, pred):  # (mean of per-trust MAEs, pooled MAE) in referral units
    err = np.abs(pred - rows["demand"].values)
    return pd.Series(err).groupby(rows["provider_code"].values).mean().mean(), err.mean()

print(f"device: {device} | rows: {len(f)} | series: {f.groupby(['provider_code','specialty_code']).ngroups}")

In [ ]:
# rung 0: no-learning baselines (predict last month / same month last year)
results = {}
te = splits["test"]
for name, col in [("naive (last month)", "lag1"), ("seasonal-naive (t-12)", "lag12")]:
    pred = np.expm1(te[col].values * te["sd"].values + te["mu"].values)
    results[name] = mae(te, pred)
    print(f"{name}: per-trust MAE {results[name][0]:.1f} | pooled {results[name][1]:.1f}")

In [ ]:
# helper: train with early selection on val (train 1 epoch at a time, keep the best val state)
# optional init -> used by fed+fine-tune so BOTH rungs share the exact same protocol
def fit_with_val(rows_tr, rows_va, epochs=40, lr=1e-3, seed=0, init=None):
    torch.manual_seed(seed)
    model = Net().to(device)
    if init is not None:
        model.load_state_dict(init)
    best, best_m = None, np.inf
    for _ in range(epochs):
        train_epochs(model, rows_tr, 1, lr)
        m, _ = mae(rows_va, predict_demand(model, rows_va))
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best)
    return model

# rung 1: local-only — each trust trains alone on its own rows
pred = np.zeros(len(te))
for t in trusts:
    m = fit_with_val(splits["train"][splits["train"]["provider_code"] == t],
                     splits["val"][splits["val"]["provider_code"] == t])
    mask = (te["provider_code"] == t).values
    pred[mask] = predict_demand(m, te[mask])
results["local-only"] = mae(te, pred)
print(f"local-only: per-trust MAE {results['local-only'][0]:.1f} | pooled {results['local-only'][1]:.1f}")

In [ ]:
# rung 2: FedAvg — broadcast global weights, 2 local epochs per client, sample-weighted average
def fedavg(rounds=100, local_epochs=2, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    global_model = Net().to(device)
    client_rows = {t: splits["train"][splits["train"]["provider_code"] == t] for t in trusts}
    weights = {t: len(client_rows[t]) / len(splits["train"]) for t in trusts}
    best, best_m, curve = None, np.inf, []
    for r in range(rounds):
        avg = None
        for t in trusts:  # each client trains from the global weights on ITS OWN rows only
            local = Net().to(device)
            local.load_state_dict(global_model.state_dict())
            train_epochs(local, client_rows[t], local_epochs, lr)
            sd = local.state_dict()
            avg = {k: weights[t] * v for k, v in sd.items()} if avg is None \
                  else {k: avg[k] + weights[t] * sd[k] for k in avg}
        global_model.load_state_dict(avg)
        m, _ = mae(splits["val"], predict_demand(global_model, splits["val"]))
        curve.append(m)
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in avg.items()}
    global_model.load_state_dict(best)
    return global_model, curve

fed, curve = fedavg()
results["FedAvg"] = mae(te, predict_demand(fed, te))
print(f"FedAvg: per-trust MAE {results['FedAvg'][0]:.1f} | pooled {results['FedAvg'][1]:.1f} "
      f"| best round {int(np.argmin(curve)) + 1}/{len(curve)}")

In [ ]:
# rung 3: FedAvg + fine-tune — IDENTICAL protocol to local-only (val-selected, same 40-epoch
# budget); the only difference is the starting point: global federated weights, not random init.
# So this rung answers exactly one question: does starting from the federation help a trust?
pred = np.zeros(len(te))
ft_models = {}
for t in trusts:
    m = fit_with_val(splits["train"][splits["train"]["provider_code"] == t],
                     splits["val"][splits["val"]["provider_code"] == t],
                     init=fed.state_dict())
    ft_models[t] = {k: v.cpu() for k, v in m.state_dict().items()}
    mask = (te["provider_code"] == t).values
    pred[mask] = predict_demand(m, te[mask])
results["FedAvg + fine-tune"] = mae(te, pred)

# rung 4: centralized — all data in one pot (the upper bound a federation cannot legally have)
central = fit_with_val(splits["train"], splits["val"])
results["centralized (upper bound)"] = mae(te, predict_demand(central, te))
for k in ["FedAvg + fine-tune", "centralized (upper bound)"]:
    print(f"{k}: per-trust MAE {results[k][0]:.1f} | pooled {results[k][1]:.1f}")

# persist models so the agentic plane (05/07) is driven by FEDERATED forecasts, not a stand-in
torch.save({"fed_global": {k: v.cpu() for k, v in fed.state_dict().items()},
            "fed_ft": ft_models, "specs": specs, "feats": FEATS},
           "../data/processed/models_04.pt")
print("saved ../data/processed/models_04.pt (global + 35 fine-tuned)")

In [ ]:
import json
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
})

table = pd.DataFrame(results, index=["per-trust MAE", "pooled MAE"]).T.round(1)
print(table.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
ax1.plot(range(1, len(curve) + 1), curve, color="#2a78d6", linewidth=2)
ax1.set_title("FedAvg convergence (val MAE by round)")
ax1.set_xlabel("round")
ax1.set_ylabel("per-trust MAE")
ax1.grid(axis="x", visible=False)

names = list(table.index)
vals = table["per-trust MAE"].values
ax2.barh(names[::-1], vals[::-1], color="#2a78d6", height=0.6)
for i, v in enumerate(vals[::-1]):
    ax2.text(v + max(vals) * 0.01, i, f"{v:.1f}", va="center", color="#0b0b0b", fontsize=8)
ax2.set_title("Test MAE by rung (lower = better)")
ax2.set_xlabel("per-trust MAE (referrals)")
ax2.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

with open("../data/processed/results_04.json", "w") as out:
    json.dump({k: {"per_trust_mae": v[0], "pooled_mae": v[1]} for k, v in results.items()}, out, indent=2)
print("saved ../data/processed/results_04.json")